In [4]:
# Minimal metrics: single-file atoms (sparsity/selectivity/overlap) + cross-run stability (directory)
import os
import glob
import torch
import numpy as np
from scipy.optimize import linear_sum_assignment


# ---------- Helpers ----------
def _orient_atoms(C: torch.Tensor) -> torch.Tensor:
    """Ensure rows are atoms (n_atoms x n_features)."""
    if C.ndim != 2:
        raise ValueError(f"Concept matrix must be 2D, got {tuple(C.shape)}")
    return C if C.size(0) <= C.size(1) else C.T.contiguous()


def _l2_row_normalize(Tx: torch.Tensor) -> torch.Tensor:
    return Tx / (Tx.norm(dim=1, keepdim=True) + 1e-12)
def l1_shrinkage_rowwise(C, lambda_l1=1e-4):
    """
    Applies L1 regularization (soft-thresholding) row-wise.
    Input:  C (n, m)
    Output: C_shrunk (n, m), same shape
    
    Formula: sign(C) * max(|C| - λ, 0)
    """
    return C.sign() * torch.clamp(C.abs() - lambda_l1, min=-0.0)

# ---------- Atom-centric metrics (single file) ----------

def compute_atom_sparsity(C, threshold=1e-1):
    """
    Robust sparsity metric using a threshold (default 1e-2).
    Computes sparsity per atom, then averages.
    Handles degenerate/constant atoms safely.
    """
    # Per-atom sparsity: fraction of near-zero values
    per_atom_sparsity = (C.abs() < threshold).float().mean(dim=1)

    # Detect collapse: all atoms have nearly the same sparsity
    if per_atom_sparsity.std() < 1e-8:
        # still return the mean, but avoids misleading behavior
        return per_atom_sparsity.mean().item()

    return per_atom_sparsity.mean().item()




def compute_atom_selectivity(C):
    """Hoyer sparsity (0..1) averaged over atoms; higher = more concentrated atoms."""
    eps = 1e-12
    n = C.size(1)
    sqrt_n = torch.sqrt(torch.tensor(float(n), device=C.device))
    l1 = C.abs().sum(dim=1)
    l2 = torch.linalg.norm(C, dim=1) + eps
    s = (sqrt_n - (l1 / l2)) / (sqrt_n - 1.0 + eps)
    return s.clamp_(0.0, 1.0).mean().item()


def compute_concept_overlap_tensor(C, normalize_rows=False, threshold=1e-3):
    """
    Mean off-diagonal cosine similarity between atoms (rows).
    Applies thresholding BEFORE similarity computation.
    Lower overlap = more disentangled concepts.
    """
    if C.ndim != 2:
        raise ValueError(f"Expected 2D concept matrix, got {tuple(C.shape)}")

    # 1. Threshold small values
    C_thr = C.clone()
    C_thr[C_thr.abs() < threshold] = 0.0

    # 2. Optional normalization
    if normalize_rows:
        Cn = C_thr / (C_thr.norm(dim=1, keepdim=True) + 1e-12)
    else:
        Cn = C_thr

    # 3. Cosine similarity matrix
    sim = Cn @ Cn.T  # shape: (K, K)
    n = sim.size(0)

    if n <= 1:
        return 0.0

    # 4. Remove diagonal
    mask = ~torch.eye(n, dtype=torch.bool, device=C.device)

    return sim[mask].mean().item()


# ---------- Cross-run stability (directory) ----------
def _mean_matched_cosine(V1: np.ndarray, V2: np.ndarray) -> float:
    """Mean matched cosine similarity between two atom banks (rows=atoms)."""
    # Normalize rows
    V1n = V1 / (np.linalg.norm(V1, axis=1, keepdims=True) + 1e-12)
    V2n = V2 / (np.linalg.norm(V2, axis=1, keepdims=True) + 1e-12)
    n = min(V1n.shape[0], V2n.shape[0])
    S = V1n[:n] @ V2n[:n].T
    row_ind, col_ind = linear_sum_assignment(-S)
    return float(S[row_ind, col_ind].mean())


def compute_cross_run_stability(base_dir: str, pattern: str = "**/concept/snmf/combined_concept_snmf_raw.pth") -> float:
    """
    Compute cross-run stability over all concept banks under base_dir.
    Stability = 1 − mean(pairwise matched cosine). Lower = worse coherence across runs.
    """
    paths = sorted(glob.glob(os.path.join(base_dir, pattern), recursive=True))
    if len(paths) < 2:
        return float("nan")

    banks = []
    for p in paths:
        try:
            blob = torch.load(p, map_location='cpu')
            C = torch.as_tensor(blob['concepts'], dtype=torch.float32)
            C = _orient_atoms(C).cpu().numpy()
            banks.append((p, C))
        except Exception:
            continue

    if len(banks) < 2:
        return float("nan")

    # Keep largest group sharing the same feature dimension
    by_d = {}
    for p, C in banks:
        by_d.setdefault(C.shape[1], []).append((p, C))
    d_key = max(by_d.keys(), key=lambda k: len(by_d[k]))
    banks = by_d[d_key]

    sims = []
    for i in range(len(banks)):
        for j in range(i + 1, len(banks)):
            _, V1 = banks[i]
            _, V2 = banks[j]
            sims.append(_mean_matched_cosine(V1, V2))

    if not sims:
        return float("nan")

    mean_sim = float(np.mean(sims))
    stability = 1.0 - mean_sim  # lower = less stable; higher mean_sim → lower stability value
    return stability

def print_atom_stats(C):
    """
    Print min, max, mean, and mean-absolute values per atom (row),
    plus global statistics.
    """
    if C.ndim != 2:
        raise ValueError(f"Expected 2D matrix, got {tuple(C.shape)}")

    # Per-atom stats
    atom_max       = C.max(dim=1).values
    atom_min       = C.min(dim=1).values
    atom_mean      = C.mean(dim=1)
    atom_mean_abs  = C.abs().mean(dim=1)

    print("=== Per-Atom Stats ===")
    for i, (mn, mx, mu, ma) in enumerate(zip(atom_min, atom_max, atom_mean, atom_mean_abs)):
        print(
            f"Atom {i:02d}: "
            f"min = {mn.item():.6f}, "
            f"max = {mx.item():.6f}, "
            f"mean = {mu.item():.6f}, "
            f"mean|x| = {ma.item():.6f}"
        )

    # Global stats
    global_min      = C.min().item()
    global_max      = C.max().item()
    global_mean     = C.mean().item()
    global_mean_abs = C.abs().mean().item()

    print("\n=== Global Stats ===")
    print(f"Global min      = {global_min:.6f}")
    print(f"Global max      = {global_max:.6f}")
    print(f"Global mean     = {global_mean:.6f}")
    print(f"Global mean|x|  = {global_mean_abs:.6f}")



# ---------- Inputs ----------
# Directory for cross-run stability
stability_dir = "/mnt/abka03/Projects/xl-vlms/outputs/gemma3n-4B_sam"
# Concept file for sparsity/selectivity/overlap
concept_file = "/mnt/abka03/Projects/xl-vlms/outputs/gemma3n-4B_sam/imnet100/concept/snmf/combined_concept_snmf_raw.pth"

# ---------- Load single concept file and compute atom metrics ----------
blob = torch.load(concept_file, map_location='cpu')
concepts = torch.as_tensor(blob['concepts'], dtype=torch.float32)
print(concepts.shape)
#concepts = _orient_atoms(concepts)#
concepts = l1_shrinkage_rowwise(concepts)

sparsity = compute_atom_sparsity(concepts)
selectivity = compute_atom_selectivity(concepts)
overlap = compute_concept_overlap_tensor(concepts)

# ---------- Compute cross-run stability from directory ----------
stability = compute_cross_run_stability(stability_dir)
print_atom_stats(concepts)
# ---------- Results ----------
results = {
    'sparsity': sparsity,
    'selectivity': selectivity,
    'stability': stability,  # cross-run (lower = worse); implicitly higher mean similarity → lower value here
    'overlap': overlap,
}
print(results)


torch.Size([9, 2048])
=== Per-Atom Stats ===
Atom 00: min = -0.153696, max = 0.182904, mean = -0.000097, mean|x| = 0.016759
Atom 01: min = -0.177724, max = 0.207078, mean = 0.000443, mean|x| = 0.015722
Atom 02: min = -0.207766, max = 0.259600, mean = 0.000450, mean|x| = 0.015571
Atom 03: min = -0.205878, max = 0.258238, mean = 0.000508, mean|x| = 0.015602
Atom 04: min = -0.154146, max = 0.248692, mean = 0.000556, mean|x| = 0.015977
Atom 05: min = -0.159461, max = 0.238337, mean = 0.000105, mean|x| = 0.015834
Atom 06: min = -0.209415, max = 0.135740, mean = 0.000268, mean|x| = 0.016002
Atom 07: min = -0.187576, max = 0.204210, mean = 0.000703, mean|x| = 0.015771
Atom 08: min = -0.208184, max = 0.214482, mean = 0.000406, mean|x| = 0.015805

=== Global Stats ===
Global min      = -0.209415
Global max      = 0.259600
Global mean     = 0.000371
Global mean|x|  = 0.015894
{'sparsity': 0.9963650107383728, 'selectivity': 0.2846616506576538, 'stability': 5.948841571812302e-05, 'overlap': 0.6322